In [113]:
import pytreenet as ptn
from copy import deepcopy
import numpy as np
import cmath
#np.random.seed(57643)

In [114]:
from qutip import coherent
def product_state(ttn, bond_dim=2 , physical_dim= 2):
    product_state = deepcopy(ttn)
    A = np.array([3/5,4/5])
    #A = np.array([0,1])
    alpha = 1
    #A = np.array(coherent(physical_dim , alpha).full())
    for node_id in product_state.nodes.keys():
        n = product_state.tensors[node_id].ndim - 1
        tensor = A.reshape((1,) * n + (physical_dim,))
        T = np.pad(tensor, n*((0, 0),) + ((0, 0),))
        product_state.tensors[node_id] = T
        product_state.nodes[node_id].link_tensor(T)  
    return product_state

In [115]:
# local physical dimension
d = 2

shapes = {
    (0, 0): (3, 5, 6, d),
    (0, 1): (3, 2, d),
    (1, 0): (5, d),
    (1, 1): (2, d)
}


sites = {
    (i, j): ptn.random_tensor_node(shapes[(i, j)], identifier=f"Site({i},{j})") for i in range(2) for j in range(2)
}

vectorized_pho = ptn.TreeTensorNetworkState()

vectorized_pho.add_root(sites[(0, 0)][0], sites[(0, 0)][1])

connections = [
    ((0, 0), (0, 1), 0, 0),
    ((0, 0), (1, 0), 1, 0),
    ((0, 1), (1, 1), 1, 0)]


for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Site({parent[0]},{parent[1]})"
    child_id = f"Site({child[0]},{child[1]})"
    vectorized_pho.add_child_to_parent(sites[child][0], sites[child][1], child_leg, parent_id, parent_leg)

vectorized_pho = product_state(vectorized_pho , bond_dim=4, physical_dim = d)

nodes = {
    (i, j): (ptn.Node(tensor=vectorized_pho.tensors[f"Site({i},{j})"] , identifier=f"Node({i},{j})"), vectorized_pho.tensors[f"Site({i},{j})"]) for i in range(2) for j in range(2)
}

vectorized_pho.add_child_to_parent(nodes[(0,0)][0], nodes[(0,0)][1], 2, "Site(0,0)", 2)

connections = [
    ((0, 0), (0, 1), 1, 0),
    ((0, 0), (1, 0), 2, 0),
    ((0, 1), (1, 1), 1, 0)]

for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Node({parent[0]},{parent[1]})"
    vectorized_pho.add_child_to_parent(nodes[child][0], nodes[child][1], child_leg, parent_id, parent_leg)

In [116]:
vectorized_pho = ptn.adjust_bra_to_ket(vectorized_pho)

In [117]:
def get_neighbors_Site(x, y, Lx, Ly):
  neighbors = []
  
  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Site({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Site({x},{y+1})")
  
  return neighbors

def get_neighbors_Node(x, y, Lx, Ly):
  neighbors = []

  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Node({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Node({x},{y+1})")
  
  return neighbors

def Liouville(t, U, gamma, m, L, Lx, Ly, d):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
    
    conversion_dict = {
        "b^dagger": creation_op,
        "b": annihilation_op,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "it * b^dagger": t*1j * creation_op,
        "it * b": t*1j * annihilation_op,
        "-iU * n * (n - 1)": -U*1j * number_op @ (number_op - np.eye(d)),
        "im*n": m*1j*number_op
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            neighbors = get_neighbors_Site(x, y, Lx, Ly)            
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "it * b^dagger", neighbor: "b"}))
                terms.append(ptn.TensorProduct({current_site: "it * b", neighbor: "b^dagger"}))
                

    
    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-iU * n * (n - 1)"}))

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "im*n"}))        
    
    H1 = ptn.Hamiltonian(terms, conversion_dict)
    
    conversion_dict = {
        "b^dagger.T": creation_op.T,
        "b.T": annihilation_op.T,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "-it * b^dagger.T": -t*1j * creation_op.T,
        "-it * b.T": -t*1j * annihilation_op.T,
        "iU * n * (n - 1).T": (U*1j * number_op @ (number_op - np.eye(d))).T,
        "-im*n.T": -m*1j* number_op.T
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            neighbors = get_neighbors_Node(x, y, Lx, Ly)
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "-it * b^dagger.T", neighbor: "b.T"}))
                terms.append(ptn.TensorProduct({current_site: "-it * b.T", neighbor: "b^dagger.T"}))

    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "iU * n * (n - 1).T"}))    

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-im*n.T"}))
            
    H2 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H2)

        
    conversion_dict = {    
    "L": np.sqrt(gamma) * L,
    "L^dagger.T": np.sqrt(gamma) * L.conj(),
    "-1/2 (L^dagger @ L) " : -1/2 * gamma * L.conj().T @ L,
    "-1/2 (L^dagger @ L).T": -1/2 * gamma * (L.conj().T @ L).T}
    terms = []
    for x in range(Lx):
        for y in range(Ly):
            out_site = f"Node({x},{y})"
            in_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({in_site: "L" , out_site: "L^dagger.T"}))
            terms.append(ptn.TensorProduct({in_site: "-1/2 (L^dagger @ L) "}))
            terms.append(ptn.TensorProduct({out_site: "-1/2 (L^dagger @ L).T"}))

    H3 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H3)
    return H1

In [118]:
def Liouville(t, U, gamma, m, L, Lx, Ly, d):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
    
    conversion_dict = {
        "b^dagger": creation_op,
        "b": annihilation_op,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "it * b^dagger": t*1j * creation_op,
        "it * b": t*1j * annihilation_op,
        "-iU * n * (n - 1)": -U*1j * number_op @ (number_op - np.eye(d)),
        "im*n": m*1j*number_op
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            neighbors = get_neighbors_Site(x, y, Lx, Ly)            
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "it * b^dagger", neighbor: "b"}))
                terms.append(ptn.TensorProduct({current_site: "it * b", neighbor: "b^dagger"}))
                

    
    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-iU * n * (n - 1)"}))

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "im*n"}))        
    
    H1 = ptn.Hamiltonian(terms, conversion_dict)
    
    conversion_dict = {
        "b^dagger.T": creation_op.T,
        "b.T": annihilation_op.T,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "-it * b^dagger.T": -t*1j * creation_op.T,
        "-it * b.T": -t*1j * annihilation_op.T,
        "iU * n * (n - 1).T": (U*1j * number_op @ (number_op - np.eye(d))).T,
        "-im*n.T": -m*1j* number_op.T
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            neighbors = get_neighbors_Node(x, y, Lx, Ly)
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "-it * b^dagger.T", neighbor: "b.T"}))
                terms.append(ptn.TensorProduct({current_site: "-it * b.T", neighbor: "b^dagger.T"}))

    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "iU * n * (n - 1).T"}))    

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-im*n.T"}))
            
    H2 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H2)

        
    conversion_dict = {    
    "L": np.sqrt(gamma) * L,
    "L^dagger.T": np.sqrt(gamma) * L.conj(),
    "-1/2 (L^dagger @ L) " : -1/2 * gamma * L.conj().T @ L,
    "-1/2 (L^dagger @ L).T": -1/2 * gamma * (L.conj().T @ L).T}
    terms = []
    for x in range(Lx):
        for y in range(Ly):
            out_site = f"Node({x},{y})"
            in_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({in_site: "L" , out_site: "L^dagger.T"}))
            terms.append(ptn.TensorProduct({in_site: "-1/2 (L^dagger @ L) "}))
            terms.append(ptn.TensorProduct({out_site: "-1/2 (L^dagger @ L).T"}))

    H3 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H3)
    return H1

In [119]:
def Number_op_total(Lx, Ly, dim=2):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(dim)
    conversion_dict = {"n": number_op , f"I{dim}": np.eye(dim)}

    terms = []
    for x in range(Lx):
        for y in range(Ly):
            in_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({in_site: "n"}))

    for x in range(Lx):
        for y in range(Ly):
            out_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({out_site: f"I{dim}"}))

    return ptn.Hamiltonian(terms, conversion_dict) 

In [120]:
N = Number_op_total(2, 2, d)
N = N.pad_with_identities(vectorized_pho, symbolic= True)
N = N.to_tensor(vectorized_pho)
N = N.operator
vectorized_pho.nodes.keys()

dict_keys(['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)'])

In [121]:
#N = np.transpose(N,(0,2,4,6,8,10,12,14,1,3,5,7,9,11,13,15))
#N = np.reshape(N,(2**4,2**4,2**4,2**4))

In [122]:
t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op
gamma = 1

H1 = Liouville(t, U, gamma, m ,L, 2, 2, d)
H1 = H1.pad_with_identities(vectorized_pho , symbolic= True)
L_fancy = ptn.TTNO.from_hamiltonian(H1, vectorized_pho)

ttno , ttno_order = L_fancy.completely_contract_tree()
ttno.shape , ttno_order 

((2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2),
 ['Site(0,0)',
  'Node(0,0)',
  'Node(0,1)',
  'Node(1,1)',
  'Node(1,0)',
  'Site(0,1)',
  'Site(1,1)',
  'Site(1,0)'])

In [123]:
ttn , ttn_order = vectorized_pho.completely_contract_tree()
initial_state_vector = ttn.reshape(-1)
ttn.shape , ttn_order , initial_state_vector.shape ,

((2, 2, 2, 2, 2, 2, 2, 2),
 ['Site(0,0)',
  'Node(0,0)',
  'Node(0,1)',
  'Node(1,1)',
  'Node(1,0)',
  'Site(0,1)',
  'Site(1,1)',
  'Site(1,0)'],
 (256,))

In [124]:
hamiltonian = np.transpose(ttno,(0,2,4,6,8,10,12,14,1,3,5,7,9,11,13,15))
hamiltonian_matrix = np.reshape(hamiltonian,(2**8,2**8))

In [125]:
exact = ptn.ExactTimeEvolution(initial_state = initial_state_vector, 
                               hamiltonian = hamiltonian_matrix, 
                               time_step_size = 0.01,
                               final_time= 1,
                               operators= N)



In [126]:
bra = exact.state.conj()
bra = bra.reshape(16,16)
N = N.reshape(16,16,16,16)
bra_op = np.tensordot(bra, N, axes = ((0,1),(2,3)))
ket = exact.state
ket = ket.reshape(16,16)
bra_op_ket = np.tensordot(bra_op, ket, axes = ((0,1),(0,1)))
bra_op_ket

array(2562500.)

In [127]:
stop

NameError: name 'stop' is not defined

In [ ]:
exact.run(evaluation_time=2)

times = exact.times()
exact_results = exact.operator_results(realise=True)[0]

In [ ]:
N.shape

In [ ]:
density_matrix = exact.state
state_vector = exact.state


In [128]:
# local physical dimension
d = 2

shapes = {
    (0, 0): (3, 5, d),
    (0, 1): (3, d) }

sites = {
    (i, j): ptn.random_tensor_node(shapes[(i, j)], identifier=f"Site({i},{j})") for i in range(1) for j in range(2)
}

vectorized_pho = ptn.TreeTensorNetworkState()

vectorized_pho.add_root(sites[(0, 0)][0], sites[(0, 0)][1])

connections = [
    ((0, 0), (0, 1), 0, 0)]


for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Site({parent[0]},{parent[1]})"
    child_id = f"Site({child[0]},{child[1]})"
    vectorized_pho.add_child_to_parent(sites[child][0], sites[child][1], child_leg, parent_id, parent_leg)

vectorized_pho = product_state(vectorized_pho , bond_dim=4, physical_dim = d)

nodes = {
    (i, j): (ptn.Node(tensor=vectorized_pho.tensors[f"Site({i},{j})"] , identifier=f"Node({i},{j})"), vectorized_pho.tensors[f"Site({i},{j})"]) for i in range(1) for j in range(2)
}

vectorized_pho.add_child_to_parent(nodes[(0,0)][0], nodes[(0,0)][1], 1, "Site(0,0)", 1)

connections = [
    ((0, 0), (0, 1), 1, 0),]

for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Node({parent[0]},{parent[1]})"
    vectorized_pho.add_child_to_parent(nodes[child][0], nodes[child][1], child_leg, parent_id, parent_leg)

    
vectorized_pho = ptn.adjust_bra_to_ket(vectorized_pho)

In [129]:
vectorized_pho.nodes

{'Site(0,0)': <pytreenet.core.node.Node at 0x1d0e0630ad0>,
 'Site(0,1)': <pytreenet.core.node.Node at 0x1d0e0630aa0>,
 'Node(0,0)': <pytreenet.core.node.Node at 0x1d0e06304a0>,
 'Node(0,1)': <pytreenet.core.node.Node at 0x1d0e0631160>}

In [130]:
creation_op, annihilation_op, number_op = ptn.bosonic_operators(2)
conversion_dict = {"n": number_op , f"I{2}": np.eye(2)}
terms = [ptn.TensorProduct({"Site(0,0)": "n"})]
terms.append(ptn.TensorProduct({"Site(0,1)": "n"}))
N = ptn.Hamiltonian(terms, conversion_dict)
N = N.pad_with_identities(vectorized_pho, symbolic= True)
N_tn = N.to_tensor(vectorized_pho)
N = N_tn.operator
print( N.shape, N_tn.node_identifiers) 

(2, 2, 2, 2, 2, 2, 2, 2) ['Site(0,0)', 'Site(0,1)', 'Node(0,0)', 'Node(0,1)']


In [131]:
ttn , ttn_order = vectorized_pho.completely_contract_tree( to_copy= True)
initial_state_vector = ttn.reshape(-1)
ttn.shape , ttn_order , initial_state_vector.shape ,

((2, 2, 2, 2), ['Site(0,0)', 'Node(0,0)', 'Node(0,1)', 'Site(0,1)'], (16,))

In [132]:
O = N.reshape(16,16)
psi = ttn.reshape(16)

In [133]:
psi.conj() @ O @ psi

799.9999999999998